# Normalización de distribuidoras

Construye `distribuidoras` (dimensión) y `peliculas_distribuidoras` (unión)
a partir de las columnas crudas de `icaa_peliculas.csv`
(`distribuidora_nacional_icaa`, `distribuidora_pdf_heuristica`) -- no de
`distribuidora_final`, que ya coalesció una sola fuente y perdió el detalle
multivalor.

**Regla de prioridad por película:**
1. Distribuidoras reales de la ficha ICAA (si hay alguna tras quitar el
   placeholder "PELICULA SIN DISTRIBUIDORA ASIGNADA")
2. Si no, la distribuidora heurística del PDF (si no es "Independent")
3. Si ninguna de las dos tiene nada real -> `Sin distribuidora`

**Identidad de distribuidora:** por nombre (caso confirmado: la misma
empresa puede tener distinto código ICAA en distintas fichas -- ej.
Universal Pictures con `(01J1481)` y `(02J22)` -- se tratan como la misma
entidad).

## 1. Librerías y carga

In [2]:
import re
import csv
from pathlib import Path

import pandas as pd

BASE = Path("..")
CSV_DIR = BASE / "3 - csv"

icaa_peliculas = pd.read_csv(CSV_DIR / "icaa_peliculas.csv", sep=';')
print(f"Películas: {len(icaa_peliculas)}")


Películas: 2052


## 2. Parseo de las dos fuentes

`PELICULA SIN DISTRIBUIDORA ASIGNADA (98J000)` puede venir MEZCLADA con
distribuidoras reales en la misma lista (33 casos) -- se filtra la entrada
placeholder, no el campo entero.

In [3]:
PLACEHOLDER_ICAA = "PELICULA SIN DISTRIBUIDORA ASIGNADA (98J000)"


def parsear_lista_icaa(valor):
    """Divide por ';' y quita la entrada placeholder, conservando el resto."""
    if pd.isna(valor) or not valor:
        return []
    entradas = [e.strip() for e in valor.split(';')]
    return [e for e in entradas if e != PLACEHOLDER_ICAA and e]


def extraer_nombre_codigo(entrada):
    """'NOMBRE, S.L. (02J570)' -> ('NOMBRE, S.L.', '02J570')."""
    m = re.match(r'^(.+?)\s*\(([\dA-Za-z]+)\)\s*$', entrada)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return entrada.strip(), None


print("✓ Funciones de parseo definidas (probadas contra 4 casos reales, incluido el mixto)")


✓ Funciones de parseo definidas (probadas contra 4 casos reales, incluido el mixto)


## 3. Aplicar regla de prioridad por película

In [4]:
registros = []
for _, row in icaa_peliculas.iterrows():
    entradas_icaa = parsear_lista_icaa(row['distribuidora_nacional_icaa'])

    if entradas_icaa:
        for entrada in entradas_icaa:
            nombre, codigo = extraer_nombre_codigo(entrada)
            registros.append({
                'pelicula_id_temp': row.name, 'nombre': nombre, 'codigo_icaa': codigo,
                'fuente': 'ficha_icaa',
            })
        continue

    heuristica = row['distribuidora_pdf_heuristica']
    if pd.notna(heuristica) and heuristica.strip() and heuristica.strip() != 'Independent':
        nombre, codigo = extraer_nombre_codigo(heuristica.strip())
        registros.append({
            'pelicula_id_temp': row.name, 'nombre': nombre, 'codigo_icaa': codigo,
            'fuente': 'pdf_heuristica',
        })
        continue

    registros.append({
        'pelicula_id_temp': row.name, 'nombre': 'Sin distribuidora', 'codigo_icaa': None,
        'fuente': 'sin_dato',
    })

peliculas_distribuidoras_temp = pd.DataFrame(registros)
print(f"✓ {len(peliculas_distribuidoras_temp)} relaciones película-distribuidora extraídas")
print()
print(peliculas_distribuidoras_temp['fuente'].value_counts())


✓ 2171 relaciones película-distribuidora extraídas

fuente
ficha_icaa        1647
sin_dato           454
pdf_heuristica      70
Name: count, dtype: int64


### Reaplicar el override manual de "Cant dels ocells, El"

Su distribuidora (`"Sagrera"`) se corrigió a mano sobre `distribuidora_final`
en la fase anterior, pero las columnas crudas (`distribuidora_pdf_heuristica`,
`distribuidora_nacional_icaa`) siguen vacías para esa fila -- si no se
reaplica aquí, esta reconstrucción la perdería y la película caería en
"Sin distribuidora".

In [5]:
mascara_sagrera = icaa_peliculas['titulo'] == 'Cant dels ocells, El'
idx_sagrera = icaa_peliculas[mascara_sagrera].index

if len(idx_sagrera):
    pelicula_id_temp_sagrera = idx_sagrera[0]
    peliculas_distribuidoras_temp = peliculas_distribuidoras_temp[
        peliculas_distribuidoras_temp['pelicula_id_temp'] != pelicula_id_temp_sagrera
    ]
    nuevo = pd.DataFrame([{
        'pelicula_id_temp': pelicula_id_temp_sagrera, 'nombre': 'Sagrera',
        'codigo_icaa': None, 'fuente': 'override_manual',
    }])
    peliculas_distribuidoras_temp = pd.concat([peliculas_distribuidoras_temp, nuevo], ignore_index=True)
    print(f"✓ Override de Sagrera reaplicado (pelicula_id_temp={pelicula_id_temp_sagrera})")
else:
    print("⚠ No se encontró la fila de 'Cant dels ocells, El' -- revisar")


✓ Override de Sagrera reaplicado (pelicula_id_temp=1814)


## 4. Construir `distribuidoras` (dimensión, dedup por nombre "base")

Dedup por nombre exacto no basta: la misma empresa aparece con razón
social completa cuando viene de ICAA (`"FILMAX, S.A."`) y con nombre corto
cuando viene del heurístico del PDF (`"Filmax"`) -- son cadenas distintas
para la misma entidad. Se normaliza quitando razón social (S.L./S.A./A.I.E.
...) y puntuación para agrupar por identidad real; el nombre que se
muestra prefiere el más completo (con código ICAA) cuando hay elección.

In [6]:
import re


def nombre_base(nombre):
    """Identidad real de la distribuidora: sin razón social ni puntuación."""
    n = nombre.upper()
    n = re.sub(r',?\s*(S\.?L\.?U?\.?|S\.?A\.?U?\.?|A\.?I\.?E\.?|S\.?COOP\.?.*)$', '', n)
    n = re.sub(r'[.,]', '', n)
    return re.sub(r'\s+', ' ', n).strip()


peliculas_distribuidoras_temp['nombre_base'] = peliculas_distribuidoras_temp['nombre'].apply(nombre_base)

# Nombre canónico por grupo: preferir el que tenga código ICAA (más completo/formal);
# si hay varios con código, el más frecuente.
def elegir_nombre_canonico(grupo):
    con_codigo = grupo[grupo['codigo_icaa'].notna()]
    candidato = con_codigo if len(con_codigo) else grupo
    return candidato['nombre'].value_counts().idxmax()

nombre_canonico_por_base = (
    peliculas_distribuidoras_temp.groupby('nombre_base')
    .apply(elegir_nombre_canonico, include_groups=False)
)

bases_unicas = sorted(nombre_canonico_por_base.index)
distribuidoras = pd.DataFrame({
    'distribuidora_id': range(1, len(bases_unicas) + 1),
    'nombre': [nombre_canonico_por_base[b] for b in bases_unicas],
})

codigo_por_base = (
    peliculas_distribuidoras_temp[peliculas_distribuidoras_temp['codigo_icaa'].notna()]
    .drop_duplicates(subset='nombre_base')
    .set_index('nombre_base')['codigo_icaa']
)
distribuidoras['codigo_icaa_referencia'] = [codigo_por_base.get(b) for b in bases_unicas]
distribuidoras['nombre_base'] = bases_unicas

print(f"✓ distribuidoras: {len(distribuidoras)} distribuidoras únicas (antes de deduplicar por base: "
      f"{peliculas_distribuidoras_temp['nombre'].nunique()})")
distribuidoras[distribuidoras['nombre_base'].isin(['FILMAX', 'BEGIN AGAIN FILMS', 'AVALON DISTRIBUCION AUDIOVISUAL'])]


✓ distribuidoras: 459 distribuidoras únicas (antes de deduplicar por base: 474)


,distribuidora_id,nombre,codigo_icaa_referencia,nombre_base
55,56,"AVALON DISTRIBUCION AUDIOVISUAL, S.L.",02J784,AVALON DISTRIBUCION AUDIOVISUAL
61,62,"BEGIN AGAIN FILMS, S.L.",02J1250,BEGIN AGAIN FILMS
166,167,"FILMAX, S.A.",02J18,FILMAX


## 5. Construir `peliculas_distribuidoras` (unión)

In [7]:
peliculas_distribuidoras = peliculas_distribuidoras_temp.merge(
    distribuidoras[['distribuidora_id', 'nombre_base']], on='nombre_base', how='left', validate='many_to_one'
)[['pelicula_id_temp', 'distribuidora_id', 'fuente']]

# Tras deduplicar por nombre_base, una misma distribuidora puede acabar dos
# veces en la misma película (ej. registrada con dos códigos ICAA distintos
# en la misma ficha, como Avalon con (02J784) y (10J2558)) -- se colapsa a
# una sola fila por (pelicula_id_temp, distribuidora_id), o rompería la
# clave primaria en MySQL.
antes = len(peliculas_distribuidoras)
peliculas_distribuidoras = peliculas_distribuidoras.drop_duplicates(
    subset=['pelicula_id_temp', 'distribuidora_id']
)
print(f"Filas colapsadas por duplicado exacto (misma película, misma distribuidora): {antes - len(peliculas_distribuidoras)}")

print(f"✓ peliculas_distribuidoras: {len(peliculas_distribuidoras)} filas")
print(f"  Películas únicas cubiertas: {peliculas_distribuidoras['pelicula_id_temp'].nunique()} / {len(icaa_peliculas)}")


Filas colapsadas por duplicado exacto (misma película, misma distribuidora): 6
✓ peliculas_distribuidoras: 2165 filas
  Películas únicas cubiertas: 2052 / 2052


## 6. Exportar

`pelicula_id_temp` es la posición dentro de `icaa_peliculas.csv` (el índice
del DataFrame) -- en `4_tablas_finales`, donde se asigna el `pelicula_id`
real (también secuencial desde la misma fuente, en el mismo orden), el
mapeo es directo: `pelicula_id = pelicula_id_temp + 1`.

In [8]:
# 'nombre_base' era solo un auxiliar interno para el merge de la sección 5 --
# no debe persistir al CSV (ni a MySQL después), así que se descarta aquí.
distribuidoras_export = distribuidoras.drop(columns=['nombre_base'])
distribuidoras_export.to_csv(CSV_DIR / "distribuidoras.csv", index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
peliculas_distribuidoras.to_csv(CSV_DIR / "peliculas_distribuidoras_temp.csv", index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)

print("✓ Exportado a CSV:")
for nombre in ['distribuidoras', 'peliculas_distribuidoras_temp']:
    ruta = CSV_DIR / f"{nombre}.csv"
    print(f"  {ruta} ({ruta.stat().st_size / 1024:.1f} KB)")


✓ Exportado a CSV:
  ..\3 - csv\distribuidoras.csv (18.5 KB)
  ..\3 - csv\peliculas_distribuidoras_temp.csv (46.2 KB)
